# WaterSoftHack 2026 — Day 2: Edge Computing (Hands-On)

**Welcome back!** Yesterday you rented a cloud supercomputer and did all the thinking there.
Today we go to the other end of the wire: **inside the gauge standing in the river**.

You will program the *brain* of a solar-powered smart stream gauge and storm-test it against a
naive design. The gauge is simulated — same Python, same JupyterHub as yesterday, no soldering —
but every number (battery, bytes, timing) uses real field hardware values.

## What you will do in the next hour

1. Build a two-week river record with one nasty flash flood in it.
2. Meet the **virtual gauge**: a battery, a sensor, a radio, and *your* code in the middle.
3. Meter the **naive streamer** - the design that transmits every reading.
4. Program the **smart gauge** - detection rules, adaptive sampling, hourly summaries, a local buffer.
5. **Cut the network in the middle of the flood** and see which design still warns the town.
6. Read the final scoreboard: messages, bytes, battery life, and warning time.

> Run each cell with **Shift+Enter**. The lecture's slogan applies everywhere here:
> **it is cheaper to think than to talk.**

> **Concept - what edge computing is, and why the cloud is not enough.** Yesterday every reading traveled to the cloud, which is fine when the sensor sits next to fiber internet. Most gauges do not. They stand in remote places on a solar panel and one bar of cell signal, and the storm that makes them important is the storm that cuts them off. **Edge computing** means doing the thinking at or near the sensor itself instead of shipping every raw byte to a distant data center. Four hard facts of the field force this: **bandwidth** is tiny (a field radio moves a text message, not a video stream), **power** is scarce (a battery and a solar panel, no wall outlet), **latency** matters (a flash-flood warning is worthless after the water arrives), and **the network fails in storms**, exactly when you need it. Putting a little intelligence inside the gauge answers all four.

---
## Part 0 — The river we will test against

A real river is boring most of the time - which is exactly the point of edge computing, but it makes
a bad *test*. So we replay a **design storm**: a two-week, minute-by-minute stage record with one
flash flood in it (quiet baseflow, then a steep rise on day 10, crest above flood stage, slow recession).

If the room's network allows, the optional cell afterwards also pulls **live** USGS stage data so you
can see what today's actual Iowa River looks like - but the storm test below is deterministic, so
everyone gets the same scoreboard.

**The design storm.** Stage in feet is a piecewise-linear hydrograph through seven fixed knots, plus a small daily wiggle and sensor noise, sampled every minute (time $\theta$ in hours):

$$s(\theta)=\operatorname{interp}\big(\theta;\ \{\theta_k\},\{s_k\}\big)+0.05\sin\tfrac{2\pi\theta}{24}+\eta,\qquad \eta\sim\mathcal{N}(0,\,0.01^{2}).$$

The storm rises at $\theta_r=24\,d_{\text{storm}}+6$ h and crests at $\theta_p=\theta_r+6$, through knots

$$(\theta_k,\,s_k)=\big[(0,2),\,(\theta_r,2),\,(\theta_r{+}2,5),\,(\theta_p,13),\,(\theta_p{+}3,9),\,(\theta_p{+}25,3),\,(24\,\text{days},2.5)\big].$$

Between two knots the value is a straight line, $\;s=s_k+(s_{k+1}-s_k)\dfrac{\theta-\theta_k}{\theta_{k+1}-\theta_k}$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def design_storm(days=14, storm_day=10, seed=42):
    """Minute-resolution stage record (feet): quiet baseflow + one flash flood."""
    n = days * 24 * 60                       # one reading per minute
    t_hr = np.arange(n) / 60.0               # time in hours
    rise = storm_day * 24 + 6                # storm rise begins 06:00 on the storm day
    peak = rise + 6                          # crest 6 hours later - a true flash flood
    keypoints_hr  = [0,   rise, rise + 2, peak, peak + 3, peak + 25, days * 24]
    keypoints_ft  = [2.0, 2.0,  5.0,      13.0, 9.0,      3.0,       2.5]
    stage = np.interp(t_hr, keypoints_hr, keypoints_ft)
    rng = np.random.default_rng(seed)
    stage += 0.05 * np.sin(2 * np.pi * t_hr / 24)          # gentle daily wiggle
    stage += rng.normal(0, 0.01, n)                        # sensor noise
    idx = pd.date_range("2026-07-06", periods=n, freq="1min")
    return pd.Series(stage, index=idx, name="stage_ft")

FLOOD_STAGE_FT = 10.0        # official flood stage for our reach
stage = design_storm()

plt.figure(figsize=(11, 4))
plt.plot(stage.index, stage.values, lw=1.2)
plt.axhline(FLOOD_STAGE_FT, color="crimson", ls="--", lw=1.5, label=f"Flood stage ({FLOOD_STAGE_FT} ft)")
plt.title("The design storm: 14 quiet days, one flash flood")
plt.ylabel("Stage (ft)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

crossing_time = stage[stage >= FLOOD_STAGE_FT].index[0]
print("The river first hits flood stage at:", crossing_time)

**Look at the shape.** Thirteen and a half days of "still about 2 feet, still fine", then a rise of
roughly **2 feet per hour** - a genuine flash flood signature. Keep two questions in mind:
how many of those ~20,000 one-minute readings are worth a radio transmission, and what happens if the
network dies during the interesting part?

### Optional: what does the real river say right now?

This cell asks the USGS API for live stage data at Iowa City (it falls back gracefully if the
network is busy). Not used in the storm test - just to connect the simulator to reality.

In [ ]:
import requests

def fetch_live_stage(site="05454500", days=7):
    url = "https://waterservices.usgs.gov/nwis/iv/"
    params = {"format": "json", "sites": site, "parameterCd": "00065",  # 00065 = gage height (ft)
              "period": f"P{days}D", "siteStatus": "all"}
    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        series = r.json()["value"]["timeSeries"]
        values = series[0]["values"][0]["value"]
        name = series[0]["sourceInfo"]["siteName"]
        d = pd.DataFrame(values)
        d["dateTime"] = pd.to_datetime(d["dateTime"])
        d["stage_ft"] = pd.to_numeric(d["value"], errors="coerce")
        d = d.dropna().set_index("dateTime")["stage_ft"]
        return d, name
    except Exception as e:
        return None, f"(live fetch skipped: {type(e).__name__})"

live, live_name = fetch_live_stage()
if live is not None:
    plt.figure(figsize=(11, 3))
    plt.plot(live.index, live.values, lw=1.5)
    plt.title(f"Live stage - {live_name}")
    plt.ylabel("Stage (ft)")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("See? Mostly boring. That is why the smart gauge mostly sleeps.")
else:
    print(live_name, "- no problem, the storm test does not need it.")

---
## Part 1 — The virtual gauge

Here is the hardware we are simulating, with realistic numbers for an ESP32-class station:

| Component | Draws | For | Cost per use |
|---|---|---|---|
| Deep sleep | 0.01 mA | always | ~0.25 mAh/**day** |
| Stage sensor read | 5 mA | 0.05 s | ~0.00007 mAh |
| Brain thinking | 40 mA | 0.01 s | ~0.0001 mAh |
| **Radio transmission** | **220 mA** | **2 s** | **~0.12 mAh** |

One battery: **10,000 mAh**. One radio message: **60 bytes**. Notice the radio costs roughly
**a thousand sensor readings** per message. That asymmetry is the whole game.

The `Gauge` class below is the simulation engine. It gives your code three abilities -
`read_sensor()`, `transmit(...)`, `local_alarm(...)` - and it keeps the honest ledger:
every microamp-hour, every byte, every message that failed because the network was down.
A gauge brain is a subclass that implements one method: `step(i, t)`, called once per simulated minute.

> **Concept - the hardware of a smart gauge.** The "brain" we program is a **microcontroller**, a tiny few-dollar computer that runs one program and sips power measured in **milliamps (mA)**. Its battery is measured in **milliamp-hours (mAh)**: a 10,000 mAh battery can supply 10,000 mA for one hour, or 1 mA for 10,000 hours. The single most important habit is the **duty cycle**, the fraction of time the device is awake. A gauge that sleeps 99% of the time lasts roughly a hundred times longer than one that never sleeps. Everything in this workshop is about spending that tiny power and bandwidth budget wisely.

**The energy accounting.** Charge equals current times time. One action drawing $I$ milliamps for $t$ seconds costs

$$q = I\cdot\frac{t}{3600}\ \text{mAh}.$$

Repeating each action $n$ times a day gives the daily draw, and the battery life on a capacity $C$ is

$$Q_{\text{day}}=\sum_{\text{actions}} n\,I\,\frac{t}{3600},\qquad \text{life}=\frac{C}{Q_{\text{day}}}.$$

Because $I_{\text{radio}}\,t_{\text{radio}}$ dwarfs every other term, $Q_{\text{day}}$ is set almost entirely by the number of transmissions.

In [ ]:
HW = dict(
    battery_mAh   = 10_000,
    sleep_mA      = 0.01,
    sensor_mA     = 5,   sensor_s = 0.05,
    mcu_mA        = 40,  mcu_s    = 0.01,
    radio_mA      = 220, radio_s  = 2.0,
    bytes_per_msg = 60,
)
mAh = lambda mA, s: mA * s / 3600.0          # milliamps for s seconds -> milliamp-hours

class Gauge:
    """Simulation engine. Subclass it and implement step(i, t)."""

    def __init__(self, stage_series, network_up=None, buffer_on_failure=False):
        self.stage = stage_series
        self.net_up = network_up if network_up is not None else np.ones(len(stage_series), bool)
        self.buffer_on_failure = buffer_on_failure
        self.used_mAh = 0.0
        self.msgs_sent = 0
        self.bytes_sent = 0
        self.buffer = []                      # queued messages during an outage
        self.cloud_log = []                   # (time, stage) pairs the CLOUD actually received
        self.alarms = []                      # (time, reason) local siren activations
        self._i = 0

    # ---- abilities your brain can use ----
    def read_sensor(self):
        self.used_mAh += mAh(HW["sensor_mA"], HW["sensor_s"]) + mAh(HW["mcu_mA"], HW["mcu_s"])
        return float(self.stage.iloc[self._i])

    def transmit(self, stage_value, kind="DATA"):
        self.used_mAh += mAh(HW["radio_mA"], HW["radio_s"])       # radio fires either way
        t = self.stage.index[self._i]
        if self.net_up[self._i]:
            self.msgs_sent += 1
            self.bytes_sent += HW["bytes_per_msg"]
            self.cloud_log.append((t, t, stage_value, kind))      # (arrived, measured, value, kind)
        elif self.buffer_on_failure:
            self.buffer.append((t, stage_value, kind))            # store-and-forward
        # else: the message is simply lost

    def flush_buffer(self):
        """Backfill queued messages once the link is up (one radio burst per message)."""
        while self.buffer and self.net_up[self._i]:
            t, v, kind = self.buffer.pop(0)
            now = self.stage.index[self._i]
            self.used_mAh += mAh(HW["radio_mA"], HW["radio_s"])
            self.msgs_sent += 1
            self.bytes_sent += HW["bytes_per_msg"]
            self.cloud_log.append((now, t, v, kind + "+BACKFILL"))

    def local_alarm(self, reason):
        self.alarms.append((self.stage.index[self._i], reason))

    # ---- the simulation loop ----
    def run(self):
        for i in range(len(self.stage)):
            self._i = i
            self.step(i, self.stage.index[i])
        days = len(self.stage) / (24 * 60)
        self.used_mAh += HW["sleep_mA"] * 24 * days               # sleep current, always on
        return self

    def report(self, label):
        days = len(self.stage) / (24 * 60)
        per_day = self.used_mAh / days
        life_days = HW["battery_mAh"] / per_day
        return {
            "design": label,
            "messages/day": round((self.msgs_sent) / days, 1),
            "radio KB/day": round(self.bytes_sent / days / 1024, 2),
            "battery mAh/day": round(per_day, 1),
            "battery life": f"{life_days/365:.1f} years" if life_days > 365 else f"{life_days/30.4:.1f} months",
        }

print("Engine ready. Battery:", HW["battery_mAh"], "mAh. One message costs",
      round(mAh(HW["radio_mA"], HW["radio_s"]), 3), "mAh - about",
      int(mAh(HW["radio_mA"], HW["radio_s"]) / (mAh(HW["sensor_mA"], HW["sensor_s"]) + mAh(HW["mcu_mA"], HW["mcu_s"]))),
      "sensor readings' worth of energy.")

---
## Part 2 — Design A: the naive streamer

The obvious design, and exactly what we built yesterday: read the sensor every minute, transmit
every reading, let the cloud do all the thinking. Three lines of brain.

> **Concept - it is cheaper to think than to talk.** Here is the fact that shapes every design choice today, and it runs opposite to laptop intuition. On battery power, **computing is nearly free and transmitting is expensive.** A microcontroller doing math draws a few tens of milliamps for a few milliseconds. Firing the radio draws a couple hundred milliamps for a couple of seconds, roughly a thousand times the energy of a sensor reading. So the battery life of a field gauge is set almost entirely by how often it transmits, not by how much it computes. Every message you avoid sending buys hours of sleep, which is why a smart gauge computes locally and sends only conclusions.

In [ ]:
class NaiveStreamer(Gauge):
    def step(self, i, t):
        value = self.read_sensor()      # every minute...
        self.transmit(value)            # ...send it. That's the whole brain.

naive = NaiveStreamer(stage).run()
naive_report = naive.report("Naive streamer")
naive_report

**Meter check.** 1,440 messages a day, ~84 KB a day, and the battery math: at roughly
175 mAh/day, a 10,000 mAh battery is gone in about **two months** - then someone drives to the
canyon with a fresh battery, forever. Also notice *what* it sent: 20,000 messages that almost all
say "still 2 feet, still fine."

On a healthy network this design does detect the flood promptly (the cloud sees every reading,
one minute late). Hold that thought - we will revisit it when the network stops being healthy.

---
## Part 3 — Design B: program the smart gauge

Now the brain from the lecture, the one you own. The behavior specification:

1. **Adaptive sampling** - read every **15 min** in quiet water, every **1 min** in event mode.
2. **Detection rules**, run on every sample, *on the device*:
   - `RAPID RISE` - the stage climbed more than **1.5 ft/hour** -> enter event mode, sound the alarm.
   - `FLOOD STAGE` - the stage crossed **10 ft** -> alarm again (the big one).
3. **The data diet** - one **hourly summary** message (min/mean/max); in event mode also a
   **situation report every 30 min**. Nothing else goes on the radio.
4. **Store-and-forward** - every reading is logged locally; if a transmission fails it is
   buffered and backfilled when the link returns.
5. Leave event mode when the water is back below 8 ft and falling.

The thresholds are the knobs at the top - after the storm test, come back and tune them.

> **Concept - the intelligence is two simple rules.** Flood onset has a signature a microcontroller can catch with two if-statements. The **threshold** rule fires when the stage crosses flood stage, which is obvious but late, because by then the flood has arrived. The **rate-of-rise** rule fires when the water is climbing faster than about 1.5 feet per hour, which catches a flash flood while the level is still below flood stage and there is time to act. Alongside the rules the gauge uses **adaptive sampling** (doze and check every 15 minutes in calm water, wake to every minute when rising) and a **data diet** (send one hourly summary instead of sixty raw readings). Simple, local, and fast beats clever, remote, and late.

**The two rules, as math.** With stage $s(t)$, flood stage $S_{\text{flood}}$, and the rise taken over the last hour, the gauge alarms when

$$s(t)\ge S_{\text{flood}}\quad\text{(threshold)}\qquad\text{or}\qquad r(t)=\frac{s(t)-s(t-1\,\text{h})}{1\,\text{h}}>r_{\text{crit}}\quad\text{(rate of rise)}.$$

The rate rule can fire while $s(t)<S_{\text{flood}}$, and that gap below flood stage is where the early warning comes from.

In [ ]:
RISE_ALERT_FT_PER_HR = 1.5     # rule 2a: rate-of-rise trigger
EVENT_EXIT_FT        = 8.0     # leave event mode below this (and falling)
QUIET_EVERY_MIN      = 15      # sampling interval, quiet mode
SITREP_EVERY_MIN     = 30      # situation reports during event mode

class SmartGauge(Gauge):
    def __init__(self, *a, **kw):
        super().__init__(*a, buffer_on_failure=True, **kw)      # store-and-forward is ON
        self.history = []            # (time, stage) samples from the last hour
        self.hour_samples = []       # samples since the last summary
        self.event_mode = False
        self.flood_alarmed = False
        self.last_sitrep_i = -10**9

    def rise_per_hour(self, t, value):
        self.history.append((t, value))
        cutoff = t - pd.Timedelta(hours=1)
        self.history = [(tt, vv) for tt, vv in self.history if tt >= cutoff]
        return value - self.history[0][1]     # rise over (up to) the past hour

    def step(self, i, t):
        self.flush_buffer()                                   # backfill whenever the link is up
        due = (i % (1 if self.event_mode else QUIET_EVERY_MIN)) == 0
        if not due:
            return                                            # stay asleep this minute
        value = self.read_sensor()
        self.hour_samples.append(value)
        rise = self.rise_per_hour(t, value)

        # ---- detection rules (this is the intelligence) ----
        if (not self.event_mode) and rise > RISE_ALERT_FT_PER_HR:
            self.event_mode = True
            self.local_alarm(f"RAPID RISE ({rise:+.1f} ft/hr)")
            self.transmit(value, "ALERT rapid rise")
        if (not self.flood_alarmed) and value >= FLOOD_STAGE_FT:
            self.flood_alarmed = True
            self.local_alarm(f"FLOOD STAGE ({value:.1f} ft)")
            self.transmit(value, "ALERT flood stage")
        if self.event_mode and value < EVENT_EXIT_FT and rise < 0:
            self.event_mode = False                           # stand down
            self.flood_alarmed = False

        # ---- the data diet ----
        if i % 60 == 0 and self.hour_samples:                 # hourly summary
            self.transmit(float(np.mean(self.hour_samples)), "SUMMARY")
            self.hour_samples = []
        elif self.event_mode and i - self.last_sitrep_i >= SITREP_EVERY_MIN:
            self.transmit(value, "SITREP")
            self.last_sitrep_i = i

smart = SmartGauge(stage).run()
smart_report = smart.report("Smart gauge")
pd.DataFrame([naive_report, smart_report]).set_index("design")

**Read the two rows slowly.** Same river, same battery, same radio - the only difference is the
brain. The smart gauge sends **~25 messages a day instead of 1,440**, about **1.5 KB instead of
84 KB**, and the battery stops being the reason you drive to the canyon: it now outlives the
hardware around it. The alarms fired too - check:

In [ ]:
for t, reason in smart.alarms:
    lead = (crossing_time - t).total_seconds() / 3600
    when = f"{lead:+.1f} h before flood stage" if lead > 0 else f"{abs(lead):.1f} h after flood stage"
    print(f"  LOCAL ALARM  {t}   {reason:28s} ({when})")
print()
print(f"The RAPID RISE rule fired while the river was still below flood stage -")
print(f"that lead time is the rate rule earning its keep.")

---
## Part 4 — The storm test: cut the network mid-flood

Now the drill the lecture promised. Storms attack the network first, so: **the cellular link goes
down one hour into the rise and stays down for 12 hours** - right through the flood-stage crossing
and the crest. This is not pessimism; it is the correlated failure the reliability slide warned about.

Same two brains. One question: **does the town get warned?**

> **Concept - store-and-forward, and why autonomy is the point.** A well-designed gauge treats a dead network the way it treats rain: expected and survivable. Every reading is logged to the on-board card first, so the radio is a courier, not the record. When the link drops, messages are **buffered** and then **backfilled** once it returns, so the cloud's record heals with no gap. And because the detection rules run on the device, the local siren still fires during the outage. The efficiency is nice, but this is the real payoff: the warning does not depend on the very infrastructure that the flood tends to destroy.

In [ ]:
# Network schedule: up everywhere, except a 12-hour outage starting 1h into the rise.
outage_start = crossing_time - pd.Timedelta(hours=3.5)   # ~1 h after the rise begins
outage_end   = outage_start + pd.Timedelta(hours=12)
net = ~((stage.index >= outage_start) & (stage.index < outage_end))
print(f"Outage: {outage_start}  ->  {outage_end}  ({12} hours, covering crossing and crest)")

naive_o = NaiveStreamer(stage, network_up=net).run()
smart_o = SmartGauge(stage,   network_up=net).run()

def cloud_flood_news(g):
    """When did the CLOUD first learn stage had reached flood stage? (arrival time counts)"""
    seen = [arrived for (arrived, measured, v, kind) in g.cloud_log
            if v >= FLOOD_STAGE_FT or "flood" in kind.lower()]
    return min(seen) if seen else None

def first_local_alarm(g):
    return g.alarms[0][0] if g.alarms else None

for label, g in [("NAIVE", naive_o), ("SMART", smart_o)]:
    alarm = first_local_alarm(g)
    news  = cloud_flood_news(g)
    print(f"\n{label}:")
    print(f"  local alarm at the gauge : {alarm if alarm else '--- none (no local intelligence) ---'}")
    print(f"  cloud learns of flood    : {news if news else '--- NEVER (readings lost in outage) ---'}")
    holes = sum(1 for (a, m, v, k) in g.cloud_log if 'BACKFILL' in k)
    print(f"  backfilled messages      : {holes}")

**This is the whole day in one printout.**

- The **naive streamer** kept firing its radio into a dead network. Its flood readings were not
  buffered, so they are *gone*: the cloud record has a 12-hour hole exactly where the flood was, and
  by the time the link returned the river had already receded below flood stage. No local alarm
  exists in that design. **The town found out from the water.**
- The **smart gauge** did not care that the network was down: the rules run on the device, so the
  siren on the bridge fired hours before flood stage, on schedule. Every reading it wanted to send
  was buffered, and after the link returned it backfilled - the cloud record *healed itself*, gap-free.

Efficiency was nice. **Autonomy is the point.**

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 4.6))
ax.plot(stage.index, stage.values, lw=1.3, color="steelblue", label="True stage")
ax.axhline(FLOOD_STAGE_FT, color="crimson", ls="--", lw=1.3, label="Flood stage")
ax.axvspan(outage_start, outage_end, color="gray", alpha=0.25, label="Network outage")

ax.plot([m for (a, m, v, k) in naive_o.cloud_log],
        [v for (a, m, v, k) in naive_o.cloud_log], ".", ms=2, color="darkorange",
        label="What the cloud received (naive)")
for t, reason in smart_o.alarms:
    ax.axvline(t, color="green", lw=1.6, alpha=0.8)
ax.plot([], [], color="green", lw=1.6, label="Smart gauge local alarms")

storm_lo = crossing_time - pd.Timedelta(hours=18)
storm_hi = crossing_time + pd.Timedelta(hours=30)
ax.set_xlim(storm_lo, storm_hi)
ax.set_title("The storm test: the naive record goes dark exactly when it matters")
ax.set_ylabel("Stage (ft)")
ax.legend(loc="upper left", fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

---
## Part 5 — The final scoreboard

Everything on one table - and these are the same numbers you saw on the lecture's payoff slide,
because that slide *is* this simulator.

In [ ]:
rows = [naive_o.report("Naive streamer"), smart_o.report("Smart gauge")]
board = pd.DataFrame(rows).set_index("design")
board["flood warned locally"]  = ["no - no local rules", f"yes - {len(smart_o.alarms)} alarms, first one early"]
board["cloud record after outage"] = ["12-hour hole, flood missing", "complete (backfilled)"]
board

---
## Wrap-up

In one hour, on a simulated-but-honest smart gauge, you:

- replayed a two-week record with a flash flood through two gauge designs,
- metered the naive streamer: **1,440 msgs/day, ~84 KB/day, ~2 months of battery**,
- programmed on-device intelligence: two detection rules, adaptive sampling, hourly summaries,
- cut the network mid-flood and watched your design **still warn the town** and **heal the record**,
- and confirmed the lecture's arithmetic: **~25 msgs/day, ~1.5 KB/day, years of battery.**

### Looking ahead to Thursday (Hybrid)

Your smart gauge is a brilliant hermit: it knows its reach perfectly and nothing else. It cannot see
the rain upstream, the basin filling, or six hours into the future - that takes the cloud model you
built on Tuesday. Thursday we wire the two together: **edge triage, cloud forecasting, one pipeline.**
Bring both notebooks.

---
## Optional challenges (if you finish early)

1. **Tune the trigger.** Set `RISE_ALERT_FT_PER_HR = 0.5` and re-run Parts 3-4. How much earlier is
   the first alarm? What is the cost (messages, battery, false-alarm risk on the daily wiggle)?
2. **Starve the gauge.** Drop the battery to 2,000 mAh in `HW`. Which design survives the two weeks?
3. **Longer outage.** Make the outage 48 hours. Does the smart gauge's backfill still heal the record?
   (Check `len(smart_o.buffer)` - what would you change so the buffer cannot overflow in a real device?)
4. **A sneakier flood.** In `design_storm`, make the rise take 24 hours instead of 8 (change `peak`).
   Does the rate rule still fire before flood stage? What rule would you add?
5. **Replay reality.** If the live fetch worked, resample `live` to 1 minute
   (`live.resample('1min').interpolate()`) and run your `SmartGauge` on it. How many messages per day
   does a *real* quiet week cost?